In [1]:
'''
LPM
'''

'\nECPF:2\n'

In [2]:
import pandas as pd
import anndata as ad

In [4]:
import perturb_lib as plib

In [41]:
df_train = ad.read_h5ad('./data/benchmark/resources/datasets/neurips-2023-data/de_train.h5ad')

In [7]:
model = plib.load_trained_model('../perturblib/.plib_cache/results/lincs_paper_lpm/LPM_9bad9756f740b28a/seed_13/model.pt')

In [39]:
df_pert = model.vocab.perturb_vocab.to_pandas()

In [80]:
df_sm = df_pert[~(df_pert['symbol'].str.contains('CRISPR'))]
df_sm.loc[:, 'symbol'] = df_sm['symbol'].str.replace('-10uM', '')

In [54]:
#CODE from perturb_lib

from pathlib import Path
import requests
from tqdm import tqdm


def download_file(url: str, save_path: Path):
    """Download helper with progress bar"""
    if save_path.exists():
        pass
        #logger.info(f"{save_path.name} found in cache.")
    else:
        #logger.info(f"Downloading {save_path.name}...")
        response = requests.get(url, stream=True, verify=False)
        total_size_in_bytes = int(response.headers.get("content-length", 0))
        block_size = 1024
        progress_bar = tqdm(total=total_size_in_bytes, unit="iB", unit_scale=True)
        temp_save_path = save_path.with_suffix(".tmp")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            with open(temp_save_path, "wb") as file:
                for data in response.iter_content(block_size):
                    progress_bar.update(len(data))
                    file.write(data)
            # shutil.move(temp_save_path, save_path)
            temp_save_path.rename(save_path)
        except Exception as e:
            #logger.error(f"Error occurred while downloading {save_path.name}: {str(e)}")
            if temp_save_path.exists():
                temp_save_path.unlink()
        finally:
            progress_bar.close()

In [55]:
# create AnnData obs object by extracting meta data
meta_url = "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/siginfo_beta.txt"
meta_path =  Path("./siginfo.txt")
download_file(meta_url, save_path=meta_path)
siginfo_df = pd.read_csv(meta_path, delimiter="\t", index_col="sig_id", low_memory=False)

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 's3.amazonaws.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
100%|██████████| 465M/465M [00:15<00:00, 29.3MiB/s] 


In [266]:
# create AnnData obs object by extracting meta data
meta_url = "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/compoundinfo_beta.txt"
meta_path =  Path("./compoundinfo_beta.txt")
download_file(meta_url, save_path=meta_path)
compoundinfo_df = pd.read_csv(meta_path, delimiter="\t", low_memory=False)

In [191]:
# create AnnData obs object by extracting meta data
meta_url = "https://s3.amazonaws.com/macchiato.clue.io/builds/LINCS2020/instinfo_beta.txt"
meta_path =  Path("./instinfo_beta.txt")
download_file(meta_url, save_path=meta_path)
instinfo_df = pd.read_csv(meta_path, delimiter="\t", low_memory=False)

In [213]:
pd.set_option("display.max_rows", 50)

In [92]:
# create AnnData obs object by extracting meta data
meta_url = 'https://lincsportal.ccs.miami.edu/dcic/api/download?path=LINCS_Data/Metadata/Small_Molecules/2020_06_16&file=SampleTable_LincsID2FacilityID2CenterBatchID_LINCS_StandardizedCmpds_LSMIDs.txt'
meta_path =  Path("./SampleTable_LincsID2FacilityID2CenterBatchID_LINCS_StandardizedCmpds_LSMIDs.txt")
download_file(meta_url, save_path=meta_path)
lsm_df = pd.read_csv(meta_path, delimiter="\t", low_memory=False)

/ictstr01/home/icb/olga.novitskaia/deg_venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'lincsportal.ccs.miami.edu'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
5.22MiB [00:00, 5.64MiB/s]


In [96]:
lsm_df = lsm_df.drop_duplicates(['SM_Center_Canonical_ID', 'SM_LINCS_ID'])

In [267]:
compoundinfo_df = compoundinfo_df[~compoundinfo_df[['pert_id', 'cmap_name']].duplicated()]

In [141]:
op3_sm = df_train.obs[['sm_name', 'sm_lincs_id', 'SMILES']].drop_duplicates().reset_index(drop=True)

In [396]:
lsm2cmap_name = {
'LSM-47134': 'SB-2343',
'LSM-47120': 'K-02288',
'LSM-1476;LSM-5290': 'proscillaridin;BRD-K11451237',
'LSM-47437': 'SGC-CBP30',
'LSM-47132': 'RN-486',
'LSM-46971': 'PRT-062607',
}

In [285]:
op3_sm_is_in_lsm_df = op3_sm[op3_sm['sm_lincs_id'].isin(lsm_df['SM_LINCS_ID'])]
op3_sm_not_in_lsm_df = op3_sm[~op3_sm['sm_lincs_id'].isin(lsm_df['SM_LINCS_ID'])]

In [290]:
lsm_df_is_in_op3_sm = lsm_df[lsm_df['SM_LINCS_ID'].isin(op3_sm_is_in_lsm_df['sm_lincs_id'])]

In [298]:
lsm_df_is_in_op3_sm_comp = lsm_df_is_in_op3_sm.merge(compoundinfo_df, how='left', left_on='SM_Center_Canonical_ID', right_on='pert_id')

In [310]:
lsm_df_is_in_op3_sm_comp_isin_df_sm = lsm_df_is_in_op3_sm_comp[lsm_df_is_in_op3_sm_comp['cmap_name'].isin(df_sm['symbol'])]

In [341]:
op3_sm_is_in_lsm_df[~op3_sm_is_in_lsm_df['sm_lincs_id'].isin(lsm_df_is_in_op3_sm_comp_isin_df_sm['SM_LINCS_ID'])]

,sm_name,sm_lincs_id,SMILES
11,Ixabepilone,LSM-43293,C/C(=C\c1csc(C)n1)[C@@H]1C[C@@H]2O[C@]2(C)CCC[...
14,O-Demethylated Adapalene,LSM-6237,O=C(O)c1ccc2cc(-c3ccc(O)c(C45CC6CC(CC(C6)C4)C5...
31,BMS-265246,LSM-46203,CCCCOc1c(C(=O)c2c(F)cc(C)cc2F)cnc2[nH]ncc12
43,Chlorpheniramine,LSM-1263,CN(C)CCC(c1ccc(Cl)cc1)c1ccccn1
61,Isoniazid,LSM-46042,NNC(=O)c1ccncc1
73,Riociguat,LSM-45758,COC(=O)N(C)c1c(N)nc(-c2nn(Cc3ccccc3F)c3ncccc23...
76,BI-D1870,LSM-45220,CC(C)CCN1c2nc(Nc3cc(F)c(O)c(F)c3)ncc2N(C)C(=O)C1C
87,GO-6976,LSM-1211,Cn1c2ccccc2c2c3c(c4c5ccccc5n(CCC#N)c4c21)CNC3=O
98,Ceritinib,LSM-36374,Cc1cc(Nc2ncc(Cl)c(Nc3ccccc3S(=O)(=O)C(C)C)n2)c...
101,Nefazodone,LSM-4031,CCc1nn(CCCN2CCN(c3cccc(Cl)c3)CC2)c(=O)n1CCOc1c...


In [365]:
lsm_df_is_in_op3_sm_comp_isin_df_sm

,SM_Center_Batch_ID,SM_SMILES_Batch,SM_Center_Canonical_ID,SM_LINCS_ID,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases
0,BRD-K53414658-001-04-1,COc1cc2nccc(Oc3ccc(NC(=O)Nc4cc(C)on4)c(Cl)c3)c...,BRD-K53414658,LSM-1005,BRD-K53414658,tivozanib,FLT1,VEGFR inhibitor,COc1cc2nccc(Oc3ccc(NC(=O)Nc4cc(C)on4)c(Cl)c3)c...,SPMVMDHWKHCIDT-UHFFFAOYSA-N,NaN
3,BRD-K90382497-001-03-2,COc1cc2ncn(-c3cc(OCc4ccccc4C(F)(F)F)c(s3)C(N)=...,BRD-K90382497,LSM-1014,BRD-K90382497,GW-843682X,PLK3,PLK inhibitor,COc1cc2ncn(-c3cc(OCc4ccccc4C(F)(F)F)c(s3)C(N)=...,JSKUWFIZUALZLX-UHFFFAOYSA-N,NaN
7,BRD-K49328571-001-06-9,Cc1nc(Nc2ncc(s2)C(=O)Nc2c(C)cccc2Cl)cc(n1)N1CC...,BRD-K49328571,LSM-1020,BRD-K49328571,dasatinib,EPHA2,KIT inhibitor,Cc1nc(Nc2ncc(s2)C(=O)Nc2c(C)cccc2Cl)cc(n1)N1CC...,ZBNZXTGUTAYRHI-UHFFFAOYSA-N,NaN
11,BRD-K92723993-001-06-7,CN1CCN(Cc2ccc(cc2)C(=O)Nc2ccc(C)c(Nc3nccc(n3)-...,BRD-K92723993,LSM-1023,BRD-K92723993,imatinib,CSF1R,KIT inhibitor,CN1CCN(Cc2ccc(cc2)C(=O)Nc2ccc(C)c(Nc3nccc(n3)-...,KTUFNOKKBVMGRW-UHFFFAOYSA-N,NaN
14,BRD-K78431006-001-10-2,C[C@@H](Oc1cc(cnc1N)-c1cnn(c1)C1CCNCC1)c1c(Cl)...,BRD-K78431006,LSM-1027,BRD-K78431006,crizotinib,ALK,ALK inhibitor,C[C@@H](Oc1cc(cnc1N)-c1cnn(c1)C1CCNCC1)c1c(Cl)...,KTEIFNKAUNYNJU-GFCCVEGCSA-N,NaN
...,...,...,...,...,...,...,...,...,...,...,...
255,BRD-K96550715-001-02-6,CC#CCn1c(nc2n(C)c(=O)n(Cc3nc(C)c4ccccc4n3)c(=O...,BRD-K96550715,LSM-45916,BRD-K96550715,linagliptin,DPP4,Dipeptidyl peptidase inhibitor,CC#CCn1c(nc2n(C)c(=O)n(Cc3nc(C)c4ccccc4n3)c(=O...,LTXREWYXXSTFRX-QGZVFWFLSA-N,NaN
256,BRD-K16803204-001-01-6,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,BRD-K16803204,LSM-45924,BRD-K16803204,filgotinib,JAK1,JAK inhibitor,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,RIJLVEAXPNLDTC-UHFFFAOYSA-N,NaN
257,BRD-K87782578-001-01-4,COCCOc1ccc(Nc2ncc(F)c(Nc3cccc(NC(=O)C=C)c3)n2)cc1,BRD-K87782578,LSM-45948,BRD-K87782578,AVL-292,BTK,BTK inhibitor,COCCOc1ccc(Nc2ncc(F)c(Nc3cccc(NC(=O)C=C)c3)n2)cc1,KXBDTLQSDKGAEB-UHFFFAOYSA-N,NaN
258,BRD-K37590257-001-02-8,COc1cccc(Nc2c(cnc3c(C)cc(cc23)S(=O)(=O)c2cccc(...,BRD-K37590257,LSM-45984,BRD-K37590257,GSK-256066,NaN,NaN,COc1cccc(Nc2c(cnc3c(C)cc(cc23)S(=O)(=O)c2cccc(...,JFHROPTYMMSOLG-UHFFFAOYSA-N,NaN


In [373]:
lsm_df_is_in_op3_sm_comp_df_sm = lsm_df_is_in_op3_sm_comp_isin_df_sm.merge(df_sm, how='left', left_on='cmap_name', right_on='symbol')

lsm_df_is_in_op3_sm_comp_df_sm = lsm_df_is_in_op3_sm_comp_df_sm.drop_duplicates(['code', 'SM_LINCS_ID'])

In [374]:
lsm_df_is_in_op3_sm_comp_df_sm

,SM_Center_Batch_ID,SM_SMILES_Batch,SM_Center_Canonical_ID,SM_LINCS_ID,pert_id,cmap_name,target,moa,canonical_smiles,inchi_key,compound_aliases,symbol,code
0,BRD-K53414658-001-04-1,COc1cc2nccc(Oc3ccc(NC(=O)Nc4cc(C)on4)c(Cl)c3)c...,BRD-K53414658,LSM-1005,BRD-K53414658,tivozanib,FLT1,VEGFR inhibitor,COc1cc2nccc(Oc3ccc(NC(=O)Nc4cc(C)on4)c(Cl)c3)c...,SPMVMDHWKHCIDT-UHFFFAOYSA-N,NaN,tivozanib,10112
1,BRD-K90382497-001-03-2,COc1cc2ncn(-c3cc(OCc4ccccc4C(F)(F)F)c(s3)C(N)=...,BRD-K90382497,LSM-1014,BRD-K90382497,GW-843682X,PLK3,PLK inhibitor,COc1cc2ncn(-c3cc(OCc4ccccc4C(F)(F)F)c(s3)C(N)=...,JSKUWFIZUALZLX-UHFFFAOYSA-N,NaN,GW-843682X,7662
2,BRD-K49328571-001-06-9,Cc1nc(Nc2ncc(s2)C(=O)Nc2c(C)cccc2Cl)cc(n1)N1CC...,BRD-K49328571,LSM-1020,BRD-K49328571,dasatinib,EPHA2,KIT inhibitor,Cc1nc(Nc2ncc(s2)C(=O)Nc2c(C)cccc2Cl)cc(n1)N1CC...,ZBNZXTGUTAYRHI-UHFFFAOYSA-N,NaN,dasatinib,8892
3,BRD-K92723993-001-06-7,CN1CCN(Cc2ccc(cc2)C(=O)Nc2ccc(C)c(Nc3nccc(n3)-...,BRD-K92723993,LSM-1023,BRD-K92723993,imatinib,CSF1R,KIT inhibitor,CN1CCN(Cc2ccc(cc2)C(=O)Nc2ccc(C)c(Nc3nccc(n3)-...,KTUFNOKKBVMGRW-UHFFFAOYSA-N,NaN,imatinib,9278
4,BRD-K78431006-001-10-2,C[C@@H](Oc1cc(cnc1N)-c1cnn(c1)C1CCNCC1)c1c(Cl)...,BRD-K78431006,LSM-1027,BRD-K78431006,crizotinib,ALK,ALK inhibitor,C[C@@H](Oc1cc(cnc1N)-c1cnn(c1)C1CCNCC1)c1c(Cl)...,KTEIFNKAUNYNJU-GFCCVEGCSA-N,NaN,crizotinib,8851
...,...,...,...,...,...,...,...,...,...,...,...,...,...
119,BRD-K96550715-001-02-6,CC#CCn1c(nc2n(C)c(=O)n(Cc3nc(C)c4ccccc4n3)c(=O...,BRD-K96550715,LSM-45916,BRD-K96550715,linagliptin,DPP4,Dipeptidyl peptidase inhibitor,CC#CCn1c(nc2n(C)c(=O)n(Cc3nc(C)c4ccccc4n3)c(=O...,LTXREWYXXSTFRX-QGZVFWFLSA-N,NaN,linagliptin,9394
120,BRD-K16803204-001-01-6,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,BRD-K16803204,LSM-45924,BRD-K16803204,filgotinib,JAK1,JAK inhibitor,O=C(Nc1nc2cccc(-c3ccc(CN4CCS(=O)(=O)CC4)cc3)n2...,RIJLVEAXPNLDTC-UHFFFAOYSA-N,NaN,filgotinib,9121
121,BRD-K87782578-001-01-4,COCCOc1ccc(Nc2ncc(F)c(Nc3cccc(NC(=O)C=C)c3)n2)cc1,BRD-K87782578,LSM-45948,BRD-K87782578,AVL-292,BTK,BTK inhibitor,COCCOc1ccc(Nc2ncc(F)c(Nc3cccc(NC(=O)C=C)c3)n2)cc1,KXBDTLQSDKGAEB-UHFFFAOYSA-N,NaN,AVL-292,102
122,BRD-K37590257-001-02-8,COc1cccc(Nc2c(cnc3c(C)cc(cc23)S(=O)(=O)c2cccc(...,BRD-K37590257,LSM-45984,BRD-K37590257,GSK-256066,NaN,NaN,COc1cccc(Nc2c(cnc3c(C)cc(cc23)S(=O)(=O)c2cccc(...,JFHROPTYMMSOLG-UHFFFAOYSA-N,NaN,GSK-256066,7635


In [463]:
op3_sm_ = op3_sm.merge(lsm_df_is_in_op3_sm_comp_df_sm[['SM_LINCS_ID', 'symbol', 'code']], how='left', left_on='sm_lincs_id', right_on='SM_LINCS_ID')

In [464]:
op3_sm_.loc[op3_sm_['symbol'].isna(), 'symbol'] = op3_sm_[op3_sm_['symbol'].isna()]['sm_lincs_id'].map(lsm2cmap_name)

In [465]:
op3_sm_.loc[:, 'symbol'] = op3_sm_['symbol'].str.split(';')

In [466]:
op3_sm_ = op3_sm_.explode('symbol').reset_index(drop=True)

In [467]:
op3_sm_.loc[:, 'symbol'] = op3_sm_['symbol'].fillna(op3_sm_.merge(df_sm, left_on='sm_name', right_on='symbol', how='left')['symbol_y'])

In [468]:
op3_sm_.loc[:, 'code'] = op3_sm_['code'].fillna(op3_sm_.merge(df_sm, on='symbol', how='left')['code_y'])

In [475]:
op3_sm_[op3_sm_['sm_name'].duplicated(False)]

,sm_name,sm_lincs_id,SMILES,SM_LINCS_ID,symbol,code
29,Proscillaridin A;Proscillaridin-A,LSM-1476;LSM-5290,C[C@@H]1O[C@@H](O[C@@H]2C=C3CC[C@@H]4[C@H](CC[...,NaN,proscillaridin,9814.0
30,Proscillaridin A;Proscillaridin-A,LSM-1476;LSM-5290,C[C@@H]1O[C@@H](O[C@@H]2C=C3CC[C@@H]4[C@H](CC[...,NaN,BRD-K11451237,688.0
33,Ruxolitinib,LSM-1139,N#CC[C@H](C1CCCC1)n1cc(-c2ncnc3[nH]ccc23)cn1,LSM-1139,ruxolitinib,9926.0
34,Ruxolitinib,LSM-1139,N#CC[C@H](C1CCCC1)n1cc(-c2ncnc3[nH]ccc23)cn1,LSM-1139,S-ruxolitinib,8155.0


In [564]:
import numpy as np
embeddings = np.zeros((len(op3_sm_['sm_name'].unique()), 128))
for i, name in enumerate(op3_sm_['sm_name'].unique()):
    if op3_sm_[op3_sm_['sm_name'] == name].shape[0] > 1:
        print(i, name)
        emb = model.perturb_embedding_layer.weight[op3_sm_[op3_sm_['sm_name'] == name]['code'].values].mean(0).numpy().astype(np.float64)
        embeddings[i] = emb
    else:    
        for j, row in enumerate(op3_sm_[op3_sm_['sm_name'] == name].iterrows()):
            if not pd.isna(row[1]['code']):
                emb = model.perturb_embedding_layer.weight[int(row[1]['code'])].numpy().astype(np.float64)
                embeddings[i] = emb
            else:
                emb = model.perturb_embedding_layer.weight.mean(0).numpy().astype(np.float64)
                embeddings[i] = emb

29 Proscillaridin A;Proscillaridin-A
32 Ruxolitinib


In [572]:
df = pd.DataFrame(op3_sm_['sm_name'].unique(), columns=['sm_name']).reset_index()

In [575]:
!mkdir -p ./data/embeddings/resources

In [576]:
df.to_csv('./data/embeddings/resources/compounds.csv', index=False)

In [577]:
with open('./data/embeddings/resources/embeddings.npy', 'wb') as f:
    np.save(f, embeddings)